# Vertical Coordinate Mismatch Report — `2026-06-26-fm`

**Scope:** FM (foundation-model) training / eval / cooldown configs in this directory.

The FM training configs (`ace-train-config-4deg-AIMIP-nc-sfno-fm-*.yaml`) pool ERA5
and SHiELD (c96) datasets that do **not** share the same hybrid sigma-pressure
vertical coordinate (`ak`/`bk`). The horizontal grid matches exactly; only the
vertical coordinate differs. Two config flags cause this mismatch to be swallowed
silently rather than raised as an error. Every number below is computed from the
source zarrs — nothing is hard-coded.

Companion to `hgtsfc_orography_diff.ipynb` (which covers the static `HGTsfc` forcing).

## Datasets involved

| Role | Vertical coord |
|---|---|
| Inference / validation / cooldown | **ERA5** |
| FM train, concat member #0 | SHiELD (amip) |
| FM train | SHiELD (ramped) |
| FM train | SHiELD (som) |
| FM train, concat member (last) | ERA5 |

Bucket root: `gs://vcm-ml-intermediate/`. `ak`/`bk` are stored as scalar variables
`ak_0..ak_8`, `bk_0..bk_8` (9 interfaces = 8 layers).

In [ ]:
import numpy as np
import xarray as xr

# GCS source paths (bucket root gs://vcm-ml-intermediate/).
paths = {
    "ERA5":          "gs://vcm-ml-intermediate/2026-04-17-era5-4deg-8layer-daily-1940-2025/2026-03-19-era5-4deg-8layer-1940-2025.zarr",
    "SHiELD-amip":   "gs://vcm-ml-intermediate/2026-01-28-vertically-resolved-c96-4deg-daily-shield-amip-ensemble-dataset/ic_0001.zarr",
    "SHiELD-ramped": "gs://vcm-ml-intermediate/2026-06-08-vertically-resolved-c96-shield-ramped-climSST-random-CO2-ensemble-fme-dataset-4deg-daily/ramped-sst-1xCO2-random-perturbation.zarr",
    "SHiELD-som":    "gs://vcm-ml-intermediate/2026-06-08-vertically-resolved-4deg-daily-c96-shield-som-ensemble-fme-dataset/1xCO2-ic_0001.zarr",
}

def load_akbk(ds):
    n = len([k for k in ds.variables if k.startswith("ak_")])
    ak = np.array([float(ds[f"ak_{i}"].values) for i in range(n)])  # Pa
    bk = np.array([float(ds[f"bk_{i}"].values) for i in range(n)])  # unitless
    return ak, bk

dsets = {name: xr.open_zarr(p, decode_times=False, chunks=None) for name, p in paths.items()}
akbk = {name: load_akbk(ds) for name, ds in dsets.items()}
for name, (ak, bk) in akbk.items():
    print(f"{name:14s} n_interfaces={len(ak)}")

In [ ]:
# All three SHiELD datasets should share one bit-identical ak/bk; ERA5 differs.
sh = [n for n in akbk if n.startswith("SHiELD")]
for i in range(1, len(sh)):
    a0, b0 = akbk[sh[0]]
    ai, bi = akbk[sh[i]]
    assert np.array_equal(a0, ai) and np.array_equal(b0, bi), f"{sh[i]} differs from {sh[0]}"
print("all SHiELD ak/bk bit-identical:", True)

ak_e, bk_e = akbk["ERA5"]
ak_s, bk_s = akbk[sh[0]]
print("ERA5 differs from SHiELD:", not (np.array_equal(ak_e, ak_s) and np.array_equal(bk_e, bk_s)))

# Horizontal grid check: ERA5 uses latitude/longitude, SHiELD uses grid_yt/grid_xt.
def latlon(ds):
    ln = "latitude" if "latitude" in ds.coords else ("lat" if "lat" in ds.coords else "grid_yt")
    lo = "longitude" if "longitude" in ds.coords else ("lon" if "lon" in ds.coords else "grid_xt")
    return np.asarray(ds[ln].values), np.asarray(ds[lo].values)

lat_e, lon_e = latlon(dsets["ERA5"])
lat_s, lon_s = latlon(dsets[sh[0]])
print("grid shape lat/lon:", lat_e.shape, lon_e.shape)
print("lat allclose:", np.allclose(lat_e, lat_s), "| lon allclose:", np.allclose(lon_e, lon_s))

## Coordinate comparison

### Raw `ak` (Pa) / `bk`

`ak` and `bk` trade off, so raw deltas look large but the effective pressure (below) is close.

In [ ]:
print(f"{'i':>2} {'ERA5 ak':>10} {'SHiELD ak':>10} {'ERA5 bk':>9} {'SHiELD bk':>9}")
for i in range(len(ak_e)):
    print(f"{i:2d} {ak_e[i]:10.1f} {ak_s[i]:10.1f} {bk_e[i]:9.4f} {bk_s[i]:9.4f}")

print(f"\nmax|Δak| = {np.abs(ak_e - ak_s).max():.1f} Pa")
print(f"max|Δbk| = {np.abs(bk_e - bk_s).max():.4f}")

### Effective reference pressure at `ps = 1013.25 hPa`  (`p = ak + bk·ps`)

Layer centers are the arithmetic mean of bounding interfaces.

In [ ]:
ps = 1013.25  # hPa

# ak is in Pa -> convert to hPa; bk is unitless.
pi_e = ak_e / 100.0 + bk_e * ps  # interface pressure, hPa
pi_s = ak_s / 100.0 + bk_s * ps
pc_e = 0.5 * (pi_e[:-1] + pi_e[1:])  # layer-center pressure, hPa
pc_s = 0.5 * (pi_s[:-1] + pi_s[1:])

print("Interface pressures (hPa):")
print(f"{'i':>2} {'ERA5':>9} {'SHiELD':>9} {'Δ':>9}")
for i in range(len(pi_e)):
    print(f"{i:2d} {pi_e[i]:9.2f} {pi_s[i]:9.2f} {pi_e[i]-pi_s[i]:9.2f}")

print("\nLayer-center pressures (hPa):")
print(f"{'k':>2} {'ERA5':>9} {'SHiELD':>9} {'Δ':>9}")
for k in range(len(pc_e)):
    print(f"{k:2d} {pc_e[k]:9.2f} {pc_s[k]:9.2f} {pc_e[k]-pc_s[k]:9.2f}")

print(f"\nmax|Δ| layer-center = {np.abs(pc_e - pc_s).max():.2f} hPa")

### ISA reference temperature per layer center (US Standard Atmosphere 1976)

Temperature a given layer-center pressure implies under the 1976 standard atmosphere,
computed by inverting the barometric formula per lapse-rate layer.

In [ ]:
g0 = 9.80665      # m/s^2
M = 0.0289644     # kg/mol
Rstar = 8.31446   # J/(mol K)

# US Standard Atmosphere 1976 base layers:
# (base geopotential height m, base temp K, lapse rate K/m, base pressure Pa)
_isa = [
    (0.0,     288.15, -0.0065,  101325.0),
    (11000.0, 216.65,  0.0,      22632.06),
    (20000.0, 216.65,  0.001,     5474.889),
    (32000.0, 228.65,  0.0028,     868.0187),
    (47000.0, 270.65,  0.0,        110.9063),
    (51000.0, 270.65, -0.0028,      66.93887),
    (71000.0, 214.65, -0.002,        3.956420),
]

def isa_temperature(p_pa):
    # Select the lowest-altitude layer whose base pressure still exceeds p.
    layer = _isa[0]
    for L_ in _isa:
        if L_[3] >= p_pa:
            layer = L_
    hb, Tb, L, Pb = layer
    if L == 0.0:
        return Tb  # isothermal layer
    # p/Pb = (T/Tb)^(-g0 M / (Rstar L))  ->  T = Tb (p/Pb)^(-Rstar L / (g0 M))
    return Tb * (p_pa / Pb) ** (-Rstar * L / (g0 * M))

isa_temperature = np.vectorize(isa_temperature)

T_e = isa_temperature(pc_e * 100.0)  # hPa -> Pa
T_s = isa_temperature(pc_s * 100.0)
dT = T_e - T_s

print(f"{'k':>2} {'p ERA5':>9} {'p SHiELD':>9} {'T ERA5':>9} {'T SHiELD':>9} {'ΔT (E-S)':>9}")
for k in range(len(pc_e)):
    print(f"{k:2d} {pc_e[k]:9.2f} {pc_s[k]:9.2f} {T_e[k]:9.2f} {T_s[k]:9.2f} {dT[k]:9.2f}")

print(f"\nmax|ΔT| = {np.abs(dT).max():.2f} K")
print(f"rms ΔT  = {np.sqrt((dT**2).mean()):.2f} K")

## How the mismatch is masked (code path)

1. **Concat base = first member.** `fme/core/dataset/concat.py:30` sets
   `self._properties = datasets[0].properties.copy()`; later members merged via
   `update()`. In the FM configs, concat member #0 is a SHiELD dataset and ERA5 is
   last, so the recorded training vertical coordinate is **SHiELD's**.
2. **`strict: false` downgrades error to warning.**
   `fme/core/dataset/properties.py:55-74`: on `vertical_coordinate != other` it
   raises only when `strict`; otherwise emits `warnings.warn`. The FM train loaders
   set `strict: false`, so the ERA5 member's differing coordinate is swallowed.
3. **Checkpoint stores the training coordinate.** `fme/ace/train/train.py:271`
   `dataset_info = train_data.dataset_info` → FM base checkpoint records SHiELD `ak`/`bk`.
4. **Eval bypasses the compatibility guard.** `fme/core/dataset_info.py:117-120`
   (`assert_compatible_with`) would raise on the coordinate diff, but
   `fme/ace/inference/evaluator.py:378` and `inference.py:348` skip it when
   `allow_incompatible_dataset: true`, which the eval suite sets on every entry.

## Effect per phase

- **FM base training:** recorded coordinate = SHiELD. With `group_weights`
  `start_value: [0.5, 0.5]`, ~50% of samples are ERA5 fields, corrected with SHiELD
  `ak`/`bk`. Corrector (`conserve_dry_air`, `moisture_budget_correction`) computes
  layer thickness `dp = Δak + ps·Δbk` with the wrong grid for those samples.
- **Eval:** FM checkpoint (SHiELD coordinate) evaluated on ERA5 data, guard suppressed.
- **Cooldown** (`*-cooldown`, `*-bestinfcooldown`): trains on ERA5-only (all SHiELD
  dropped). `parameter_init.weights_path` loads **weights only**, so
  `dataset_info` is re-derived from ERA5 → cooldown coordinate = **ERA5**, matching
  the eval target. The coordinate therefore flips SHiELD → ERA5 between base training
  and cooldown while inheriting SHiELD-trained weights.

In [ ]:
# Severity summary — all figures pulled from the computed arrays above.
print("Severity: SMALL. Layer identity preserved; shared top & surface interfaces.")
print(f"  max|Δbk|                = {np.abs(bk_e - bk_s).max():.4f}")
print(f"  max|Δ| layer-center p   = {np.abs(pc_e - pc_s).max():.2f} hPa")
print(f"  max|ΔT| (ISA reference) = {np.abs(dT).max():.2f} K")
print(f"  rms ΔT                  = {np.sqrt((dT**2).mean()):.2f} K")
print(f"  top interface shared    = {np.isclose(pi_e[0], pi_s[0], atol=3.0)} (both near model top)")
print(f"  surface interface equal = {np.isclose(pi_e[-1], pi_s[-1])}")

## Recommendation

1. Harmonize the vertical coordinate before pooling — regrid the ERA5-8layer dataset
   onto SHiELD's `ak`/`bk` (or vice versa) so every training/eval/cooldown dataset
   shares one coordinate.
2. Until harmonized, keep `strict: false` / `allow_incompatible_dataset: true` only
   with awareness that they are masking this specific mismatch; do not rely on them to
   catch a future, larger coordinate error.
3. If harmonizing is not feasible, consider ordering ERA5 first in the concat so the
   recorded coordinate matches the eval/cooldown target, removing the base→cooldown flip.